In [1]:
import requests
import time

def get_hospital_address_and_zip(hospital_name, api_key):
    # Target URL for Geocoding API
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    
    # Combine hospital name and city to make the search highly accurate
    search_query = f"{hospital_name}, TX"
    
    params = {
        "address": search_query,
        "key": api_key
    }
    
    try:
        response = requests.get(base_url, params=params)
        data = response.json()
        
        if data["status"] == "OK":
            # Extract the first match found by Google
            result = data["results"][0]
            formatted_address = result["formatted_address"]
            
            # Loop through address components to find the 'postal_code'
            zip_code = "UNKNOWN"
            for component in result["address_components"]:
                if "postal_code" in component["types"]:
                    zip_code = component["long_name"]
                    break
                    
            return zip_code, formatted_address
            
        elif data["status"] == "ZERO_RESULTS":
            return "UNKNOWN", "UNKNOWN"
        else:
            print(f"API Error Status: {data['status']}")
            return "UNKNOWN", "UNKNOWN"
            
    except Exception as e:
        print(f"Request failed: {e}")
        return "UNKNOWN", "UNKNOWN"


In [ ]:
# --- Set API key ---
API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"

In [ ]:
import pandas as pd

df = pd.read_csv('Facility_type1q2019_tab.txt',sep='\t') # read facilities file
df.drop(columns=['Unnamed: 12'],inplace=True) # drop that weird column
df['ZIP'] = '00000' # set default zip code

for idx, provider in enumerate(df.loc[0:,'PROVIDER_NAME']):    
    zip_code, full_address = get_hospital_address_and_zip(provider, API_KEY)
    print(idx)
    print(f"Hospital: {provider}")
    print(f"Zip Code: {zip_code}")
    print(f"Full Address: {full_address}")
    print("-" * 40)    
    df.loc[idx,'ZIP'] = zip_code
    # Pause slightly between requests to respect basic rate limits
    time.sleep(0.1)

df = df[['THCIC_ID','PROVIDER_NAME','ZIP']]
df.to_csv('Facility_type1q2019_tab_ZIPCODE.csv',index=False)

0
Hospital: Austin State Hospital
Zip Code: 78751
Full Address: 4110 Guadalupe St, Austin, TX 78751, USA
----------------------------------------
1
Hospital: Big Spring State Hospital
Zip Code: 79720
Full Address: 1901 US-87, Big Spring, TX 79720, USA
----------------------------------------
2
Hospital: UT Medical Branch Hospital
Zip Code: 77555
Full Address: 301 University Blvd, Galveston, TX 77555, USA
----------------------------------------
3
Hospital: Rio Grande State Center
Zip Code: 78552
Full Address: 1401 S Rangerville Rd, Harlingen, TX 78552, USA
----------------------------------------
4
Hospital: UT MD Anderson Cancer Center
Zip Code: 77030
Full Address: 1515 Holcombe Blvd, Houston, TX 77030, USA
----------------------------------------
5
Hospital: Kerrville State Hospital
Zip Code: 78028
Full Address: 721 Thompson Dr, Kerrville, TX 78028, USA
----------------------------------------
6
Hospital: Rusk State Hospital
Zip Code: 75785
Full Address: 805 N Dickinson Dr, Rusk, TX 